In [1]:
from pydantic import BaseModel, Field

from dotenv import load_dotenv
import dspy

In [22]:
class DraftSyntheticEmailDescription(dspy.Signature):
    """Given a ClinicalTrials.gov study ID and a role (eg CRA, CTM),
    create descriptions of email threads that the person with the role (at the sponsor level) would be expected to be help resolve.
    """

    study_overview: str = dspy.InputField(desc="The ClinicalTrials.gov study overview")
    role: str = dspy.InputField(
        desc="The role of the person who would be expected to take action in response to the email thread"
    )
    num_threads: int = dspy.InputField(
        desc="The number of email threads to create descriptions for"
    )

    # Outputs
    email_descriptions: list[str] = dspy.OutputField(
        desc="A list of descriptions of the email threads that the role would be expected to help resolve. Don't number threads."
    )


class Email(BaseModel):
    """An email message."""

    from_address: str
    to_addresses: list[str]
    cc_addresses: list[str]
    subject: str
    timestamp: str
    body: str
    attachments: list[str]

    message_id: str
    in_reply_to: str | None


class DraftSyntheticEmailThread(dspy.Signature):
    """Given a description of an email thread and a role of a person to include, draft a synthetic email thread that includes the person with the given role (this person is always at the sponsor organization)."""

    email_description: str = dspy.InputField(desc="The description of the email thread to draft")
    role: str = dspy.InputField(
        desc="The role of the person (at the sponsor level) that must take action (reply to email, redirect email, confirm follow up, schedule meeting, etc) in the email thread. This person is always at the sponsor organization."
    )

    email_thread: list[Email] = dspy.OutputField(desc="The synthetic email thread")
    role_descriptions: list[str] = dspy.OutputField(
        desc="Map of email addresses to (organization, role) pairs in the email thread. Organization can be sponsor, CRO or site."
    )

In [23]:
load_dotenv("../.env")
lm = dspy.LM("gemini/gemini-2.5-flash-lite", temperature=0.5, cache=True)
dspy.settings.configure(lm=lm, track_usage=True)

In [24]:
STUDY_OVERVIEW = """
            Brief Summary

            This is a Phase 1/2, multicenter, randomized, placebo-controlled, double-blind study to evaluate the safety, tolerability, pharmacokinetics (PK), and pharmacodynamics (PD) of single and multiple doses of DNL593 in two parts followed by an optional open-label extension (OLE) period.

            Part A will evaluate the safety, tolerability, PK, and PD of single doses of DNL593 in healthy male and healthy female participants of nonchildbearing potential. Part B will evaluate the safety, tolerability, PK, and PD of multiple doses of DNL593 in participants with frontotemporal dementia (FTD) over 25 weeks. Part B will be followed by Part C, an optional 18-month OLE period available for all participants who complete Part B.
            Official Title
            A Phase 1/2, Multicenter, Randomized, Placebo-Controlled, Double Blind Single Dose and Multiple Dose Study to Evaluate the Safety, Tolerability, Pharmacokinetics, and Pharmacodynamics of DNL593 in Healthy Participants and Participants With Frontotemporal Dementia Followed by an Open-Label Extension
            Conditions
            Frontotemporal Dementia
            Intervention / Treatment

                Drug: DNL593
                Drug: Placebo

            Sponsor: Denali Therapeutics
          """

ROLE = "Clinical Research Associate (CRA)"

NUM_THREADS = 10

In [25]:
email_descriptions_predict = dspy.Predict(DraftSyntheticEmailDescription)
email_descriptions = email_descriptions_predict(
    study_overview=STUDY_OVERVIEW, role=ROLE, num_threads=NUM_THREADS
).email_descriptions

email_descriptions

['Thread regarding the initiation of site monitoring visits for DNL593 study, including scheduling, access to study documents, and initial observations from the CRA.',
 'Email exchange concerning the resolution of data queries identified during source data verification for the DNL593 study. The CRA is following up on outstanding queries with site staff.',
 'Discussion about the enrollment status of the DNL593 study. The CRA is reporting on recruitment numbers, identifying potential barriers, and proposing solutions to improve enrollment.',
 'Communication regarding the investigational product (DNL593 and placebo) accountability at study sites. The CRA is confirming receipt, storage, and dispensing records.',
 'Thread focused on adverse event (AE) and serious adverse event (SAE) reporting for the DNL593 study. The CRA is ensuring timely and accurate reporting by the sites.',
 'Emails related to protocol deviations identified during monitoring visits for the DNL593 study. The CRA is work

In [26]:
email_thread_predict = dspy.Predict(DraftSyntheticEmailThread)
email_thread = email_thread_predict(email_description=email_descriptions[0], role=ROLE).email_thread

email_thread

[Email(from_address='sponsor.coordinator@sponsor.org', to_addresses=['dr.evans@site.org'], cc_addresses=[], subject='Initiation of Site Monitoring Visits - DNL593 Study', timestamp='2023-10-15T09:00:00Z', body="Dear Dr. Evans,\n\nI hope this email finds you well.\n\nI am writing to confirm the initiation of site monitoring visits for the DNL593 study at your site. Our Clinical Research Associate (CRA), Sarah Chen, will be conducting the first monitoring visit on October 26th, 2023. \n\nSarah will require access to the following study documents:\n- Investigator Site File (ISF)\n- Source Documents\n- Pharmacy Records\n- Laboratory Reports\n\nPlease let us know if there are any specific protocols or requirements we need to be aware of for accessing these documents during the visit. \n\nWe will follow up with a separate email to coordinate the exact timing and any specific needs for Sarah's visit.\n\nThank you for your cooperation.\n\nSincerely,\n[Sponsor Contact Name]\n[Sponsor Organizati